# Uplift-моделирование

Notebook визуализирует только сохранённые результаты моделей на реальной клиентской витрине. Синтетические результаты не создаются.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

ARTIFACT_DIR = Path('../data/processed/uplift_modeling')
paths = {
    'comparison': ARTIFACT_DIR / 'uplift_model_comparison.parquet',
    'curves': ARTIFACT_DIR / 'uplift_curves.parquet',
    'deciles': ARTIFACT_DIR / 'uplift_deciles.parquet',
    'scores': ARTIFACT_DIR / 'uplift_customer_scores.parquet',
}
missing = [str(path) for path in paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Нет результатов uplift-моделей. Выполните python -m src.models.model_evaluation. ' + str(missing)
    )

## Сравнение моделей на holdout

In [ ]:
comparison = pl.read_parquet(paths['comparison']).to_pandas()
display(comparison)

## Qini curve

In [ ]:
curves = pl.read_parquet(paths['curves']).to_pandas()
sns.lineplot(
    data=curves, x='targeted_fraction',
    y='cumulative_incremental_purchases', hue='model'
)
baseline = curves[curves['model'] == curves['model'].iloc[0]]
plt.plot(
    baseline['targeted_fraction'], baseline['qini_random_baseline'],
    color='black', linestyle='--', label='Случайный таргетинг'
)
plt.title('Qini curve на holdout')
plt.legend()
plt.show()

## Uplift по децилям

In [ ]:
deciles = pl.read_parquet(paths['deciles']).to_pandas()
display(deciles)
sns.barplot(data=deciles, x='decile', y='observed_uplift', hue='model')
plt.axhline(0, color='black', linewidth=1)
plt.title('Observed uplift по децилям holdout')
plt.show()

## Распределение predicted uplift после final fit

In [ ]:
scores = pl.read_parquet(paths['scores']).to_pandas()
score_columns = ['s_learner_predicted_uplift', 't_learner_predicted_uplift']
sns.histplot(scores[score_columns], bins=40, element='step', stat='density', common_norm=False)
plt.axvline(0, color='black', linestyle='--')
plt.title('Распределение predicted uplift')
plt.show()